# 1. 에이전트 루프

에이전트는 모델, 도구, 루프, 정지 조건으로 구성된다.
도구 셋과 루프 하나를 만들고, 정지 조건과 트레이스를 붙인다.

### 공통 설정

1주차와 같은 블록이다. 노트북끼리 코드를 공유하지 않는다.

In [2]:
import json
import os
import time
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]

client = OpenAI()

PRICES = {MODEL: (2.50, 15.00)}  # 1M 토큰당 (입력, 출력) USD
USAGE = []


def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp

### 도구는 실제 파일을 만진다

한국어 위키백과에서 대한민국 임시정부와 관련된 문서 50편을 모은 코퍼스다(`data/corpus/graph/`).
1주차 Part 5 GraphRAG(`20`~`23`)가 그래프를 만들 때 쓴 문서와 같다. 모두 합쳐 797,748자이고 문서 하나는 중앙값 5,883자다.
김구, 안창호, 한국독립당처럼 같은 인물과 단체가 여러 문서에 걸쳐 나온다. 그래서 "몇 편에 나오는가" 같은 질문을 만들기 쉽고, 정답은 코드로 다시 세어 확인할 수 있다.
문서 한 편이 `.md` 파일 하나이고, 파일 이름이 곧 문서 제목이다. 문서별 판(revision)은 `SOURCES.tsv` 에 있다.
모델이 이미 외우고 있는 문서라는 점이 문제가 되는데, 이것은 뒤의 '답이 나왔다고 도구를 쓴 것은 아니다'에서 다룬다.
도구 셋은 그 위에서 목록, 검색, 읽기를 한다. 파이썬 함수 셋이 전부이고 특별한 것은 없다.

In [3]:
CORPUS = Path("../../../../data/corpus/graph")


# 코퍼스에 있는 문서 제목을 모두 돌려준다. 파일 이름에서 .md 를 뗀 것이 제목이다
def list_docs():
    return [f.stem for f in sorted(CORPUS.glob("*.md"))]


# 낱말이 들어 있는 문서마다 제목과 등장 횟수를 돌려준다. 없는 문서는 빠진다
def grep_docs(pattern):
    hits = []
    for f in sorted(CORPUS.glob("*.md")):
        n = f.read_text(encoding="utf-8").count(pattern)
        if n:
            hits.append({"title": f.stem, "count": n})
    return hits


print(len(list_docs()), "편")
print(grep_docs("조선민족혁명당")[:3])

0 편
[]


`read_doc`은 없는 제목을 받으면 예외를 던진다. 이 예외가 루프에서 어떻게 되는지 뒤에서 본다.

In [ ]:
# 제목에 맞는 문서 본문의 앞 2,000자를 돌려준다. 없는 제목이면 예외를 던진다
def read_doc(title):
    f = CORPUS / f"{title}.md"
    if not f.exists():
        raise FileNotFoundError(title)
    return f.read_text(encoding="utf-8")[:2000]


print(read_doc("안중근")[:80].replace("\n", " "))

### 모델에 넘기는 것은 함수가 아니라 설명이다 (TOOL REGISTRY)

모델은 위 함수를 볼 수 없다. 이름과 설명과 인자 스키마만 본다.
그래서 여기 적는 문장이 곧 프롬프트다. `2`에서 이 문장을 다룬다.
</br>
description : llm에게 tool에 대한 설명 전달

In [ ]:
TOOLS = [
    {"type": "function", "name": "list_docs", "strict": True,
     "description": "코퍼스에 있는 문서 제목을 모두 돌려준다.",
     "parameters": {"type": "object", "properties": {}, "required": [],
                    "additionalProperties": False}},
    {"type": "function", "name": "grep_docs", "strict": True,
     "description": "낱말이 들어 있는 문서의 제목과 등장 횟수를 돌려준다.",
     "parameters": {"type": "object",
                    "properties": {"pattern": {"type": "string", "description": "찾을 낱말"}},
                    "required": ["pattern"], "additionalProperties": False}},
    {"type": "function", "name": "read_doc", "strict": True,
     "description": "문서 한 편의 본문 앞부분을 돌려준다.",
     "parameters": {"type": "object",
                    "properties": {"title": {"type": "string",
                                             "description": "문서 제목. list_docs 가 준 것 그대로"}},
                    "required": ["title"], "additionalProperties": False}},
]

호출 이름을 함수로 잇는 표와 이 노트북의 과제다.

In [ ]:
REGISTRY = {"list_docs": list_docs, "grep_docs": grep_docs, "read_doc": read_doc}
TASK = "이 문서 묶음에서 한국독립당과 조선민족혁명당을 모두 언급한 문서는 몇 편인가"

### 에이전트 루프

첫 요청에 도구 정의를 붙여 보낸다. 모델은 답 대신 호출 아이템을 내놓는다.

In [ ]:
r = keep(client.responses.create(model=MODEL, input=TASK, tools=TOOLS))
print("아이템", [i.type for i in r.output])
print("output_text", repr(r.output_text))

호출 아이템마다 함수를 실행하고 `function_call_output`으로 되돌린다. 호출 아이템이 없을 때까지 이것을 반복하는 것이 루프다.

In [ ]:
def run(task, tools=TOOLS, max_turns=6): # 예산을 6턴으로 지정
    r = keep(client.responses.create(model=MODEL, input=task, tools=tools))
    for _ in range(max_turns):
        calls = [i for i in r.output if i.type == "function_call"]
        if not calls:
            return r.output_text
        out = [{"type": "function_call_output", "call_id": c.call_id,
                "output": json.dumps(REGISTRY[c.name](**json.loads(c.arguments)), ensure_ascii=False)}  # ensure_ascii = False (한글이 깨지지 않게끔 아스키 변환 x)
               for c in calls]
        r = keep(client.responsesㅁ.create(model=MODEL, previous_response_id=r.id,
                                         tools=tools, input=out))
    return "예산 소진"

과제를 끝까지 돌린다.

In [ ]:
print(run(TASK))

### 서버 관리 대화

`previous_response_id`를 사용하면 기록을 직접 관리하지 않아도 되지만 컨텍스트의 내용과 크기를 제어할 수 없다. `3`에서는 컨텍스트를 직접 관리한다.

### 한 턴에 여러 번 부른다

호출 아이템이 하나라고 가정한 코드는 곧 깨진다. 위 루프가 배열을 순회하는 이유다.

In [ ]:
r = keep(client.responses.create(model=MODEL, tools=TOOLS,
                                 input="한국독립당과 조선민족혁명당이 각각 몇 편에 나오는지 센다"))
print("아이템", [i.type for i in r.output])

호출 아이템마다 함수 이름과 인자를 읽는다.

In [ ]:
for i in r.output:
    if i.type == "function_call":
        print(i.call_id, i.name, i.arguments)

### 도구 오류는 예외가 아니라 입력이다

`read_doc`은 없는 제목을 받으면 예외를 던진다. 루프 안에서 던지면 루프가 죽는다.

In [ ]:
try:
    print(run("'김구 (독립운동가)' 문서의 첫머리를 읽는다"))
except Exception as e:
    print("루프가 죽었다:", type(e).__name__, e)

### 오류 문자열을 되돌리면 모델이 복구한다

무엇이 잘못됐는지와 다음에 무엇을 하라는 것을 한 문장에 담는다.
오류 메시지가 곧 복구 지시문이다.
</br> exception을 입력으로 처리해서, 모델이 알맞은 행위를 할 수 있게

In [ ]:
def call_tool(name, args):
    try:
        return json.dumps(REGISTRY[name](**args), ensure_ascii=False)
    except FileNotFoundError as e: # exception을 catch해서 handle 하게끔 처리 => 최종적으로 사용자에게 메시지가 감
        return f"error: '{e}' 라는 제목의 문서가 없다. list_docs 로 제목을 확인하고 그대로 다시 부른다."
    except Exception as e:
        return f"error: {type(e).__name__} {e}"

없는 제목으로 직접 불러 본다. 예외 대신 문자열이 돌아온다.

In [ ]:
print(call_tool("read_doc", {"title": "김구 (독립운동가)"}))

### 정지 조건이 없으면 에이전트가 아니라 사고다

예산은 셋이다. 반복, 토큰, 시간. 셋 중 하나라도 없으면 언젠가 그 방향의 오버플로우가 발생한다(무한반복, 과다 토큰 소진, 너무 긴 실행 시간).  
그리고 무엇이 멈췄는지를 돌려줘야 한다. "답이 없다"와 "예산이 끝났다"는 다른 사건이다. 

In [ ]:
def run_budgeted(task, tools=TOOLS, max_turns=6, max_tokens=60000, max_seconds=90):
    t0, used, trace = time.perf_counter(), 0, []
    r = keep(client.responses.create(model=MODEL, input=task, tools=tools))
    for turn in range(1, max_turns + 1):
        used += r.usage.total_tokens
        calls = [i for i in r.output if i.type == "function_call"]
        trace.append({"턴": turn, "도구": [c.name for c in calls],
                      "누적토큰": used, "초": round(time.perf_counter() - t0, 1)})
        if not calls:
            return r.output_text, "완료", trace
        if used > max_tokens:
            return None, "토큰 예산", trace
        if time.perf_counter() - t0 > max_seconds:
            return None, "시간 예산", trace
        out = [{"type": "function_call_output", "call_id": c.call_id,
                "output": call_tool(c.name, json.loads(c.arguments))} for c in calls]
        r = keep(client.responses.create(model=MODEL, previous_response_id=r.id,
                                         tools=tools, input=out))
    return None, "반복 예산", trace

같은 과제를 돌린다. 정지 사유와 답이 따로 돌아온다.

In [ ]:
answer, why, trace = run_budgeted(TASK)
print(why, "|", answer)

트레이스에는 턴마다 부른 도구, 누적 토큰, 경과 시간이 남는다.

In [ ]:
for row in trace:
    print(f"  턴 {row['턴']}  {row['초']:5.1f}초  {row['누적토큰']:6,d} 토큰  {row['도구']}")

앞에서 루프를 죽였던 과제를 다시 돌린다. 오류 문자열을 받은 모델이 `list_docs`로 제목을 확인하고 다시 읽는지 트레이스로 본다.

In [ ]:
answer, why, trace = run_budgeted("'김구 (독립운동가)' 문서의 첫머리를 읽는다")
print(why, "|", str(answer)[:80])
print([row["도구"] for row in trace])

### 예산을 좁히면 무엇이 달라지는가

같은 과제에 반복 예산을 매우 적게 준다. 답이 없는 것이 아니라 예산이 끝난 것이다.
이 구분이 없으면 실패 원인을 모델 탓으로 돌리게 된다.

In [ ]:
answer, why, trace = run_budgeted(TASK, max_turns=1)
print(why, "|", answer)
print("마지막 턴에서 부르려던 것:", trace[-1]["도구"])

### 답이 나왔다고 도구를 쓴 것은 아니다

이 코퍼스는 위키백과라서 모델이 이미 외우고 있다(`data/corpus/README.md`).
도구를 빼고 두 가지 질문을 던져본다.

In [ ]:
for kind, q in [("단일홉", "강우규가 1919년에 저격하려 한 인물은 누구인가"),
                ("집계", "한국어 위키백과의 대한민국 임시정부 관련 문서 50편 중"
                         " 한국독립당을 언급한 문서는 몇 편인가. 숫자만 답한다.")]:
    r = keep(client.responses.create(model=MODEL, input=q))
    print(f"{kind}  {r.output_text}")

집계 질문의 실제 값과 비교한다. 단일홉은 도구 없이 맞고, 집계는 세어 보지 않고 낸 숫자다.
둘 다 최종 답만 보면 도구를 썼는지 알 수 없다. 그래서 트레이스가 필요하다.

In [ ]:
print("실제 값:", len(grep_docs("한국독립당")), "편")

### 하네스 구성 요소

| 부분 | 이 노트북의 어디 |
|---|---|
| Runtime | `run_budgeted`의 for 문과 세 가지 예산 |
| ToolRegistry | `TOOLS`, `REGISTRY`, `call_tool`의 오류 처리 |
| Session | `previous_response_id`. 지금은 서버가 들고 있다 |
| Trace | `trace` 리스트 |
| Policy | 없다 : 툴 호출 관련 제한하는게 없음 |

Policy가 없어 `read_doc`에 상위 경로를 전달하면 코퍼스 밖 파일도 읽을 수 있다.
인자를 검증하는 자리는 `2`에서 넣고, 승인 게이트는 Part 4 `14`에서 붙인다.

In [ ]:
print(read_doc("../README")[:60].replace("\n", " "))

### 비용 확인

`keep`으로 모아 둔 usage에 단가를 곱해 합산한다. 이 노트북을 한 번 실행하는 비용이다.

In [ ]:
total = 0.0
for name, u in USAGE:
    p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
    if p:
        total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]

print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${total:.4f}")